# 🏦 Predictive Modeling: Banking Anomaly Detection (PoC)
---
**Objective:** Membangun arsitektur *Machine Learning* klasifikasi (Decision Tree & Random Forest Ensemble) untuk mengidentifikasi perilaku transaksi anomali secara *real-time* berdasarkan *ground truth* dari proses *Customer Segmentation* sebelumnya.

## 1. Environment Setup & Data Ingestion
Mempersiapkan *environment* komputasi dan memuat dataset hasil *clustering* yang telah dianotasi (memiliki label `Target`) sebagai fondasi pelatihan algoritma *Supervised Learning*.

In [31]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import classification_report
import joblib

In [32]:
# Gunakan dataset hasil clustering yang memiliki fitur Target
df = pd.read_csv("../data/clustered_customers_inverse.csv")
df.head()

,TransactionAmount,TransactionType,Location,Channel,CustomerAge,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance,CustomerAge_Bin,Target
0,14.09,Debit,San Diego,ATM,70.0,Doctor,81.0,1.0,5112.21,Tinggi,1
1,376.24,Debit,Houston,ATM,68.0,Doctor,141.0,1.0,13758.91,Tinggi,0
2,126.29,Debit,Mesa,Online,19.0,Student,56.0,1.0,1122.35,Rendah,1
3,184.50,Debit,Raleigh,Online,26.0,Student,25.0,1.0,8569.06,Rendah,1
4,92.15,Debit,Oklahoma City,ATM,18.0,Student,172.0,1.0,781.68,Rendah,1


## 2. Feature Engineering & Pre-processing
Melakukan transformasi fitur kategorikal menjadi representasi matriks numerik menggunakan metode *One-Hot Encoding* (`pd.get_dummies`). Proses ini esensial untuk memastikan kompatibilitas seluruh dimensi data dengan kalkulasi jarak dan pembelahan simpul (*node splitting*) pada algoritma *tree-based*.

In [33]:
categorical_cols = list(df.select_dtypes(include=['object']).columns)

# Gunakan 'pd.get_dummies' untuk meMelakukan OneHotEncoding
df_encoded = pd.get_dummies(
    df,
    columns = categorical_cols,
    drop_first = True
)

# Tampilkan 5 baris pertama untuk memverifikasi hasilnya
df_encoded.head()

,TransactionAmount,CustomerAge,TransactionDuration,LoginAttempts,AccountBalance,Target,TransactionType_Debit,Location_Atlanta,Location_Austin,Location_Baltimore,...,Location_Tucson,Location_Virginia Beach,Location_Washington,Channel_Branch,Channel_Online,CustomerOccupation_Engineer,CustomerOccupation_Retired,CustomerOccupation_Student,CustomerAge_Bin_Sedang,CustomerAge_Bin_Tinggi
0,14.09,70.0,81.0,1.0,5112.21,1,True,False,False,False,...,False,False,False,False,False,False,False,False,False,True
1,376.24,68.0,141.0,1.0,13758.91,0,True,False,False,False,...,False,False,False,False,False,False,False,False,False,True
2,126.29,19.0,56.0,1.0,1122.35,1,True,False,False,False,...,False,False,False,False,True,False,False,True,False,False
3,184.50,26.0,25.0,1.0,8569.06,1,True,False,False,False,...,False,False,False,False,True,False,False,True,False,False
4,92.15,18.0,172.0,1.0,781.68,1,True,False,False,False,...,False,False,False,False,False,False,False,True,False,False


## 3. Stratified Data Splitting
Memisahkan dataset ke dalam *Training Set* (80%) dan *Testing Set* (20%). Menerapkan parameter `stratify=y` untuk mengunci dan menjaga keseimbangan rasio target kelas (*normal vs anomaly*) pada kedua himpunan data guna mencegah model mengalami bias mayoritas.

In [34]:
# Menggunakan train_test_split() untuk meMelakukan pembagian dataset.

# Buat 'X' dengan menghapus 'Target' dari 'df_encoded' dan gunakan 'axis=1' untuk menandakan drop kolom.
X = df_encoded.drop('Target', axis=1)

# Buat 'y' dengan HANYA memilih kolom 'Target'.
y = df_encoded['Target']

# Panggil fungsi untuk membagi data.
#  - Gunakan 'stratify=y' agar proporsi kelas di train/test set sama.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42,
    stratify = y
)



print("Jumlah data total: ",len(X))
print("Jumlah data Melatih: ",len(X_train))
print("Jumlah data test: ",len(X_test))

Jumlah data total:  1945
Jumlah data Melatih:  1556
Jumlah data test:  389


## 4. Model Architecture Initialization
Membangun dan melatih (*training*) algoritma prediktif menggunakan **Decision Tree Classifier** sebagai *baseline model* untuk memetakan aturan keputusan (*decision rules*) dasar dalam mendeteksi anomali.

In [35]:
# 1. Buat (instantiate) objek model Decision Tree
#    Gunakan 'random_state=42' agar hasilnya konsisten
decision_tree_model = DecisionTreeClassifier(random_state=42)

# 2. Melatih (fit) model dengan data training (X_train dan y_train)
decision_tree_model.fit(X_train, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [36]:
# Menyimpan Model
joblib.dump(decision_tree_model, '../models/decision_tree_model.h5')

['../models/decision_tree_model.h5']

## 5. Advanced Ensemble Modeling
Meningkatkan ketahanan dan kapabilitas prediktif dengan membangun arsitektur **Random Forest Classifier**. Pendekatan *ensemble learning* ini menggabungkan ratusan pohon keputusan untuk menekan risiko *overfitting* dan menaikkan akurasi secara fundamental.

In [37]:
# Melatih model menggunakan algoritma klasifikasi scikit-learn selain Decision Tree.
# Buat (instantiate) objek model baru
new_model = RandomForestClassifier(random_state=42)

# Melatih (fit) model dengan data training (X_train dan y_train)
new_model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


### 5.1 Model Evaluation & Comparison
Mengevaluasi dan membandingkan performa model *baseline* (Decision Tree) dan *ensemble* (Random Forest) menggunakan metrik pengukuran klasifikasi yang presisi: Akurasi, Precision, Recall, dan F1-Score pada data pengujian.

In [38]:
# Menampilkan hasil evaluasi akurasi, presisi, recall, dan F1-Score pada seluruh algoritma yang sudah dibuat.

# Buat prediksi pada data 'X_test' menggunakan kedua model
y_pred_dt = decision_tree_model.predict(X_test)
y_pred_new = new_model.predict(X_test)

# Tampilkan classification_report untuk Decision Tree
print("Decision Tree Performance")
print(classification_report(y_test, y_pred_dt))

print("="*50)

# Tampilkan classification_report untuk New Model
print("New Model Performance")
print(classification_report(y_test, y_pred_new))



Decision Tree Performance
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       196
           1       1.00      1.00      1.00       193

    accuracy                           1.00       389
   macro avg       1.00      1.00      1.00       389
weighted avg       1.00      1.00      1.00       389

New Model Performance
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       196
           1       1.00      1.00      1.00       193

    accuracy                           1.00       389
   macro avg       1.00      1.00      1.00       389
weighted avg       1.00      1.00      1.00       389



In [39]:
# Menyimpan Model Selain Decision Tree

joblib.dump(new_model, '../models/explore_random-forest_classification.h5')

['../models/explore_random-forest_classification.h5']

## 6. Hyperparameter Optimization (GridSearchCV)
Mengekstraksi batasan maksimal performa dari model prediktif melalui *Hyperparameter Tuning*. Menerapkan **5-Fold Cross-Validation** untuk mencari dan menguji secara komprehensif kombinasi parameter yang paling optimal (`max_depth`, `min_samples_split`, `criterion`).

In [40]:
# Melakukan Hyperparameter Tuning dan Melatih ulang.

# Menentukan Hyperparameter yang akan di-tuning
params = {'max_depth':[3, 5, 10, None],
          'min_samples_split':[2, 5, 10],
          'criterion':['gini', 'entropy'],}

# Buat (instantiate) objek dari algoritma tuning
#  - 'estimator': Model yang akan di-tuning
#  - 'params': Hyperparameter yang sudah ditentukan
new_model_tuned = GridSearchCV(
    estimator = DecisionTreeClassifier(random_state=42),
    param_grid= params,
    cv = 5,
    scoring = 'accuracy'
)

# Melatih objek model dengan data training (X_train dan y_train)
new_model_tuned.fit(X_train, y_train)


,estimator,DecisionTreeC...ndom_state=42)
,param_grid,"{'criterion': ['gini', 'entropy'], 'max_depth': [3, 5, ...], 'min_samples_split': [2, 5, ...]}"
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,criterion,'gini'


In [41]:
# Menampilkan hasil evaluasi akurasi, presisi, recall, dan F1-Score pada algoritma yang sudah dituning.

# Buat prediksi pada 'X_test' Gunakan model yang sudah di-tuning
y_pred_tuning = new_model_tuned.predict(X_test)

# Tampilkan classification_report untuk model yang sudah di-tuning
print("Tuned Model Performance")
print(classification_report(y_test, y_pred_tuning))



Tuned Model Performance
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       196
           1       1.00      1.00      1.00       193

    accuracy                           1.00       389
   macro avg       1.00      1.00      1.00       389
weighted avg       1.00      1.00      1.00       389



## 7. Final Model Export
Menyimpan *engine Machine Learning* hasil optimasi (dengan Akurasi 100% pada *simulated environment*) ke dalam format `.h5`. Model prediktif ini kini siap untuk proses *deployment* ke dalam ekosistem server perbankan sesungguhnya.

In [42]:
# Menyimpan Model hasil tuning
joblib.dump(new_model_tuned, '../models/tuning_classification.h5')

['../models/tuning_classification.h5']

End of Code